In [0]:
from pyspark.sql.functions import *
from pyspark.sql import Window

In [0]:
%sql
create table if not exists upi_baseline_stats as 
select * from upi_pipeline_metrics

In [0]:
baseline_df = spark.read.table("workspace.default.upi_baseline_stats")

window = Window.partitionBy()

zscore_df = (baseline_df\
.withColumn("txn_bronze_count_mean",mean("txn_bronze_count").over(window))\
.withColumn("txn_bronze_count_stddev",stddev("txn_bronze_count").over(window))\

.withColumn("settlement_bronze_count_mean",mean("settlement_bronze_count").over(window))\
.withColumn("settlement_bronze_count_stddev",stddev("settlement_bronze_count").over(window))\

.withColumn("txn_silver_count_mean",mean("txn_silver_count").over(window))\
.withColumn("txn_silver_count_stddev",stddev("txn_silver_count").over(window))\

.withColumn("settlement_silver_count_mean",mean("settlement_silver_count").over(window))\
.withColumn("settlement_silver_count_stddev",stddev("settlement_silver_count").over(window))\

.withColumn("bronze_duration_mean",mean("bronze_duration").over(window))\
.withColumn("bronze_duration_stddev",stddev("bronze_duration").over(window))\

.withColumn("silver_duration_mean",mean("silver_duration").over(window))\
.withColumn("silver_duration_stddev",stddev("silver_duration").over(window))\

.withColumn("gold_duration_mean",mean("gold_duration").over(window))\
.withColumn("gold_duration_stddev",stddev("gold_duration").over(window))\

.withColumn("total_duration_mean",mean("total_duration").over(window))\
.withColumn("total_duration_stddev",stddev("total_duration").over(window))\

.withColumn("settlement_rescued_count_threshold",max(col("settlement_rescued_count")).over(window))\

.withColumn("txn_silver_reject_count_threshold",max(col("txn_silver_reject_count")).over(window))\

.withColumn("txn_rejection_rate_threshold",max(col("txn_rejection_rate")).over(window))\

.withColumn("gold_discrep_unsettled_success_count_threshold",max(col("gold_discrep_unsettled_success_count")).over(window))\

.withColumn("gold_discrep_stale_pending_count_threshold",max(col("gold_discrep_stale_pending_count")).over(window))\

.withColumn("orphan_record_count_threshold",max(col("orphan_record_count")).over(window))\

.withColumn("discrepancy_rate_threshold",max(col("discrepancy_rate")).over(window))\
)\
    .select("txn_bronze_count_mean","txn_bronze_count_stddev","settlement_bronze_count_mean","settlement_bronze_count_stddev","txn_silver_count_mean","txn_silver_count_stddev","settlement_silver_count_mean","settlement_silver_count_stddev","bronze_duration_mean","bronze_duration_stddev","silver_duration_mean","silver_duration_stddev","gold_duration_mean","gold_duration_stddev","total_duration_mean","total_duration_stddev","settlement_rescued_count_threshold","txn_silver_reject_count_threshold","txn_rejection_rate_threshold","gold_discrep_unsettled_success_count_threshold","gold_discrep_stale_pending_count_threshold","orphan_record_count_threshold","discrepancy_rate_threshold").dropDuplicates()

 
zscore_df.write.saveAsTable("workspace.default.statistics")

In [0]:

cross_joined = spark.sql("""
                         select * from workspace.default.upi_pipeline_metrics
                         cross join
                         workspace.default.statistics
                         """)

cross_joined = (
    cross_joined
    .withColumn("txn_bronze_count_zscore",(col("txn_bronze_count")-col("txn_bronze_count_mean"))/col("txn_bronze_count_stddev"))\
    .withColumn("settlement_bronze_count_zscore",(col("settlement_bronze_count")-col("settlement_bronze_count_mean"))/col("settlement_bronze_count_stddev"))\
    .withColumn("txn_silver_count_zscore",(col("txn_silver_count")-col("txn_silver_count_mean"))/col("txn_silver_count_stddev"))\
    .withColumn("settlement_silver_count_zscore",(col("settlement_silver_count")-col("settlement_silver_count_mean"))/col("settlement_silver_count_stddev"))\
    .withColumn("bronze_duration_zscore",(col("bronze_duration")-col("bronze_duration_mean"))/col("bronze_duration_stddev"))\
    .withColumn("silver_duration_zscore",(col("silver_duration")-col("silver_duration_mean"))/col("silver_duration_stddev"))\
    .withColumn("gold_duration_zscore",(col("gold_duration")-col("gold_duration_mean"))/col("gold_duration_stddev"))\
    .withColumn("total_duration_zscore",(col("total_duration")-col("total_duration_mean"))/col("total_duration_stddev"))
)

cross_joined.write.mode("append").saveAsTable("workspace.default.upi_pipeline_metrics_with_statistics")


cross_joined.display()


In [0]:
df = spark.read.table("workspace.default.upi_pipeline_metrics_with_statistics").orderBy(col("run_date").desc()).limit(1)

# df = spark.read.table("workspace.default.upi_pipeline_metrics_with_statistics").filter(col("silver_duration_zscore")==-2.4165876607615635)

cols = df.collect()
row_dict = cols[0].asDict()
import builtins
z_scores = ["txn_bronze_count","settlement_bronze_count","txn_silver_count","settlement_silver_count","bronze_duration","silver_duration","gold_duration","total_duration"]

Thresholds = ["settlement_rescued_count","txn_silver_reject_count","txn_rejection_rate","gold_discrep_unsettled_success_count","gold_discrep_stale_pending_count","orphan_record_count","discrepancy_rate"]

anomalies = {}

for i in z_scores:
    if builtins.abs(row_dict[f"{i}_zscore"]) > 2*0.9:
        anomalies[i]={
            "value": row_dict[i],
            "zscore": row_dict[f"{i}_zscore"],
            "mean": row_dict[f"{i}_mean"],
            "stddev": row_dict[f"{i}_stddev"]
        }

for j in Thresholds:
    if row_dict[j] >= row_dict[f"{j}_threshold"]:
        anomalies[j]={
            "value": row_dict[j],
            "threshold": row_dict[f"{j}_threshold"]
        }

print(anomalies)

In [0]:
from openai import OpenAI

GROQ_TOKEN = dbutils.secrets.get(scope="credentials", key="GROQ_TOKEN")

client = OpenAI(
    api_key=GROQ_TOKEN,
    base_url="https://api.groq.com/openai/v1"
)

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
            {"role" : "system" , "content" : "You are a helpful AI assistant who monitors the data pipeline anomalies and provides a short narrative summary. Use only provided metrics and do not speculate and do not hallucinate."},
            {"role": "user", "content": f"Provide a short narrative summary for the metrics {anomalies}"}
    ],
    max_tokens=500
)

print(response.choices[0].message.content)